# 04 — Verify all models on CPU

Checks that YOLO, ByteTrack, and Face models load and run on **CPU** end-to-end on a dummy frame.


In [ ]:
from pathlib import Path
import sys

def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "main.py").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Could not find the project root from the current notebook location.")

ROOT = find_project_root()
sys.path.insert(0, str(ROOT))

checks = {
    "yolo11n.pt": ROOT / "models" / "yolo" / "yolo11n.pt",
    "yolo11n.onnx": ROOT / "models" / "yolo" / "yolo11n.onnx",
    "bytetrack.yaml": ROOT / "models" / "tracker" / "bytetrack.yaml",
    "face pack": ROOT / "models" / "face" / "models" / "buffalo_s",
}
for k, p in checks.items():
    ok = p.exists()
    print(("✓" if ok else "✗"), k, "->", p)


In [ ]:
import numpy as np
import sys
from pathlib import Path
sys.path.insert(0, str(ROOT))

from src.detection.person_yolo import PersonDetector
from src.tracking.bytetrack import ByteTracker
from src.recognition.face_engine import FaceEngine
from src.events.entry_exit import EntryExitEngine
from src.events.store import EventsStore
from src.overlay.draw import OverlayRenderer

frame = np.full((720, 1280, 3), 40, dtype=np.uint8)

# 1) Person detector
yolo_pt = ROOT / "models" / "yolo" / "yolo11n.pt"
if yolo_pt.exists():
    det = PersonDetector(weights=str(yolo_pt), device="cpu", imgsz=416)
    dets = det.detect(frame)
    print(f"✓ YOLO person detect OK ({len(dets)} dets on blank)")
else:
    print("✗ Run notebook 01 first")
    dets = []

# 2) Tracker
tr = ByteTracker()
tracks = tr.update(dets)
print(f"✓ ByteTrack OK ({len(tracks)} tracks)")

# 3) Face
face_root = ROOT / "models" / "face"
try:
    fe = FaceEngine(root=str(face_root), pack="buffalo_s")
    faces = fe.detect_and_embed(frame)
    print(f"✓ Face engine OK ({len(faces)} faces on blank)")
except Exception as e:
    print("✗ Face engine:", e)
    faces = []

# 4) Events path (no cross expected)
store = EventsStore(str(ROOT / "data" / "db" / "events.db"))
ee = EntryExitEngine(line_norm={"x1": 0.45, "y1": 0.1, "x2": 0.45, "y2": 0.9})
events = ee.update(tracks, frame.shape, store)
print(f"✓ EntryExit engine OK ({len(events)} events)")

# 5) Overlay
ov = OverlayRenderer()
vis = ov.draw(frame, tracks, faces, ee.line_norm, ee.counts)
print("✓ Overlay OK", vis.shape)
print("\nAll CPU checks finished.")
